# UMUD — Apo Geometry Guard (Phase 3)

**GPU notebook** — compares MT geometry extraction using:

1. **Baseline geometry**: use predicted mask style (`tag_apo_style`) and the normal geometry pipeline.
2. **Guarded geometry**: if predicted apo coverage is extremely high, derive a boundary mask
   (mask minus eroded mask) and compute geometry from that boundary as a `line` mask.

Writes:
- `/kaggle/working/apo_geometry_guard_compare.csv`



## Configuration


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

IMG_SIZE = 256
PRED_COV_GUARD_THRESH = 0.95

ERODE_KERNEL = 5
N_GALLERY = 8
RANDOM_SEED = 42

MASK_OVERLAY_ALPHA = 0.55
APO_OVERLAY_COLOR = (255, 140, 0)

FIG_DIR = Path("/kaggle/working/figures/apo_geometry_guard")
FIG_DIR.mkdir(parents=True, exist_ok=True)

APO_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-apo-mounted-phase-3/apo_baseline.pkl"
)

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
TEST_DIR = COMPETITION_DIR / "test_images_v2/test_set_v2"


In [ ]:
from __future__ import annotations

import cv2
import numpy as np
import pandas as pd
from fastai.vision.all import PILImage, load_learner
from PIL import Image
from tqdm.auto import tqdm

IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}


def list_test_images(directory: Path) -> list[Path]:
    files = [
        p
        for p in directory.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS and p.name != "Thumbs.db"
    ]
    # Prefer .tif when both .tif and .png exist for same stem
    by_stem: dict[str, Path] = {}
    for p in sorted(files):
        stem = p.stem
        if stem not in by_stem or p.suffix.lower() in {".tif", ".tiff"}:
            by_stem[stem] = p
    return sorted(by_stem.values(), key=lambda p: p.name)


def load_gray(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.array(im)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    return arr.astype(np.uint8)


def resize_image(img: np.ndarray, size: int) -> np.ndarray:
    return np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR), dtype=np.uint8)


def resize_mask_to(mask: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    if mask.shape == (target_h, target_w):
        return (mask > 0).astype(np.uint8)
    src = (mask > 0).astype(np.uint8) * 255
    out = Image.fromarray(src).resize((target_w, target_h), Image.NEAREST)
    return (np.array(out) > 0).astype(np.uint8)


def tensor_to_mask(pred) -> np.ndarray:
    if hasattr(pred, "cpu"):
        pred = pred.cpu().numpy()
    arr = np.asarray(pred)
    if arr.ndim == 3:
        arr = arr.argmax(axis=0)
    return (arr > 0).astype(np.uint8)


def open_rgb_256(img_native: np.ndarray) -> PILImage:
    small = resize_image(img_native, IMG_SIZE)
    rgb = np.stack([small, small, small], axis=-1).astype(np.uint8)
    return PILImage.create(rgb)


def tag_apo_style(coverage: float) -> str:
    return "region" if coverage >= APO_REGION_THRESHOLD else "line"


def invert_mask(mask: np.ndarray) -> np.ndarray:
    return (1 - mask).astype(np.uint8)


def effective_apo_mask(mask: np.ndarray, style: str) -> tuple[np.ndarray, str]:
    if style == "region":
        return invert_mask(mask), "inverted_region"
    return mask, "raw_line"


def find_apo_contours(mask: np.ndarray, min_area_frac: float = 0.0003) -> list[np.ndarray]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    min_area = mask.size * min_area_frac
    big = [c for c in contours if cv2.contourArea(c) >= min_area]
    big.sort(key=lambda c: cv2.boundingRect(c)[1])
    return big


def pick_superficial_deep(contours: list[np.ndarray], min_sep_px: int = 15):
    if len(contours) < 2:
        return None, None, len(contours)
    sup = contours[0]
    _, y0, _, _ = cv2.boundingRect(sup)
    deep = None
    for c in contours[1:]:
        _, y1, _, _ = cv2.boundingRect(c)
        if y1 >= y0 + min_sep_px:
            deep = c
            break
    if deep is None:
        deep = contours[min(2, len(contours) - 1)]
    return sup, deep, len(contours)


def edge_polyline(contour: np.ndarray, which: str = "bottom", n_bins: int = 60):
    pts = contour.reshape(-1, 2)
    if len(pts) < 3:
        return np.array([]), np.array([])
    x_min, x_max = pts[:, 0].min(), pts[:, 0].max()
    if x_max <= x_min:
        return pts[:, 0].astype(float), pts[:, 1].astype(float)
    edges = np.linspace(x_min, x_max, n_bins + 1)
    xs_out, ys_out = [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        in_bin = pts[(pts[:, 0] >= lo) & (pts[:, 0] < hi)]
        if len(in_bin) == 0:
            continue
        y = in_bin[:, 1].max() if which == "bottom" else in_bin[:, 1].min()
        xs_out.append((lo + hi) / 2.0)
        ys_out.append(float(y))
    return np.array(xs_out), np.array(ys_out)


def fit_line(xs: np.ndarray, ys: np.ndarray):
    if len(xs) < 2:
        return None
    return np.poly1d(np.polyfit(xs, ys, 1))


def line_angle_deg(line) -> float:
    return float(np.degrees(np.arctan(line[1]))) if line is not None else np.nan


def mt_from_apo_edges(sup_line, deep_line, x_left: float, x_right: float) -> float:
    if sup_line is None or deep_line is None or x_right <= x_left:
        return np.nan
    thirds = [x_left + (x_right - x_left) * t for t in (1 / 6, 3 / 6, 5 / 6)]
    dists = [abs(float(deep_line(x) - sup_line(x))) for x in thirds]
    return float(np.mean(dists))


def fascicle_pca(mask: np.ndarray) -> dict | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < 3:
        return None
    coords = np.column_stack([xs.astype(float), ys.astype(float)])
    centered = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    direction = vh[0]
    projections = centered @ direction
    return {
        "length_px": float(projections.max() - projections.min()),
        "angle_deg": float(np.degrees(np.arctan2(direction[1], direction[0]))),
    }


def acute_angle_deg(a1: float, a2: float) -> float:
    d = abs(a1 - a2) % 180.0
    return float(min(d, 180.0 - d))


def apo_geometry_attempt(apo_mask: np.ndarray, style: str) -> dict:
    eff, method = effective_apo_mask(apo_mask, style)
    contours = find_apo_contours(eff)
    sup_c, deep_c, n_contours = pick_superficial_deep(contours)
    out = {
        "apo_method": method,
        "n_contours": n_contours,
        "mt_px": np.nan,
        "deep_angle_deg": np.nan,
        "mt_fail_reason": "ok",
        "sup_line": None,
        "deep_line": None,
        "sup_xs": None,
        "sup_ys": None,
        "deep_xs": None,
        "deep_ys": None,
    }
    if len(contours) == 0:
        out["mt_fail_reason"] = "no_contours"
        return out
    if n_contours < 2 or sup_c is None or deep_c is None:
        out["mt_fail_reason"] = "single_contour"
        return out
    sup_x, sup_y = edge_polyline(sup_c, which="bottom")
    deep_x, deep_y = edge_polyline(deep_c, which="top")
    sup_line = fit_line(sup_x, sup_y)
    deep_line = fit_line(deep_x, deep_y)
    out.update(sup_line=sup_line, deep_line=deep_line, sup_xs=sup_x, sup_ys=sup_y, deep_xs=deep_x, deep_ys=deep_y)
    if sup_line is None or deep_line is None:
        out["mt_fail_reason"] = "line_fit_fail"
        return out
    if len(sup_x) == 0 or len(deep_x) == 0:
        out["mt_fail_reason"] = "empty_edge_polyline"
        return out
    x_left = max(sup_x.min(), deep_x.min())
    x_right = min(sup_x.max(), deep_x.max())
    if x_right <= x_left:
        out["mt_fail_reason"] = "no_x_overlap"
        return out
    out["mt_px"] = mt_from_apo_edges(sup_line, deep_line, x_left, x_right)
    out["deep_angle_deg"] = line_angle_deg(deep_line)
    if np.isnan(out["mt_px"]):
        out["mt_fail_reason"] = "mt_compute_nan"
    return out


def apo_geometry_from_mask(apo_mask: np.ndarray, style: str) -> dict:
    primary = apo_geometry_attempt(apo_mask, style)
    primary["geometry_path"] = primary["apo_method"]
    if not np.isnan(primary["mt_px"]):
        return primary
    if style == "region":
        fallback = apo_geometry_attempt(apo_mask, "line")
        if not np.isnan(fallback["mt_px"]):
            fallback["apo_method"] = f"{primary['apo_method']}+fallback_line"
            fallback["geometry_path"] = "fallback_line"
            fallback["mt_fail_reason_primary"] = primary["mt_fail_reason"]
            return fallback
    primary["mt_fail_reason_primary"] = primary["mt_fail_reason"]
    return primary


def derive_geometry(fasc_mask: np.ndarray, apo_mask: np.ndarray, apo_style: str) -> dict:
    apo = apo_geometry_from_mask(apo_mask, apo_style)
    fpca = fascicle_pca(fasc_mask)
    out = {
        "pa_deg": np.nan,
        "fl_px": np.nan,
        "mt_px": apo["mt_px"],
        "apo_method": apo.get("apo_method"),
        "geometry_path": apo.get("geometry_path"),
        "n_contours": apo.get("n_contours"),
        "mt_fail_reason": apo.get("mt_fail_reason"),
        "mt_fail_reason_primary": apo.get("mt_fail_reason_primary"),
        "apo_cov": float(apo_mask.mean()),
        "apo_fg_pixels": int(apo_mask.sum()),
        "fasc_cov": float(fasc_mask.mean()),
        "fasc_fg_pixels": int(fasc_mask.sum()),
    }
    if fpca is not None:
        out["fl_px"] = fpca["length_px"]
        ref = apo["deep_angle_deg"] if not np.isnan(apo["deep_angle_deg"]) else 0.0
        out["pa_deg"] = acute_angle_deg(fpca["angle_deg"], ref)
    return out


In [ ]:
def overlay(img_gray: np.ndarray, mask: np.ndarray, color=APO_OVERLAY_COLOR, alpha=MASK_OVERLAY_ALPHA):
    rgb = np.stack([img_gray, img_gray, img_gray], axis=-1).astype(np.float32)
    tint = np.zeros_like(rgb)
    tint[..., 0], tint[..., 1], tint[..., 2] = color
    sel = mask.astype(bool)
    rgb[sel] = (1 - alpha) * rgb[sel] + alpha * tint[sel]
    return rgb.astype(np.uint8)


def boundary_from_mask(mask01: np.ndarray, erode_kernel: int = ERODE_KERNEL):
    import cv2

    m = (mask01.astype(np.uint8) > 0).astype(np.uint8)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_kernel, erode_kernel))
    er = cv2.erode(m, k, iterations=1)
    b = (m.astype(np.int16) - er.astype(np.int16)) > 0
    return b.astype(np.uint8)


In [ ]:
assert APO_MODEL_PATH.exists(), f"Missing apo model: {APO_MODEL_PATH}"
assert TEST_DIR.exists(), f"Missing test dir: {TEST_DIR}"

apo_learn = load_learner(APO_MODEL_PATH)
test_paths = list_test_images(TEST_DIR)
print(f"Test images: {len(test_paths)}")


In [ ]:
rows = []
for path in tqdm(test_paths, desc="geometry guard compare"):
    img_native = load_gray(path)
    h, w = img_native.shape

    pil = open_rgb_256(img_native)
    _, apo_t, _ = apo_learn.predict(pil)
    apo_native = resize_mask_to(tensor_to_mask(apo_t), h, w)

    pred_cov = float(apo_native.mean())
    style_base = tag_apo_style(pred_cov)
    base_geo = apo_geometry_from_mask(apo_native, style_base)

    guard_applied = pred_cov >= PRED_COV_GUARD_THRESH
    if guard_applied:
        bmask = boundary_from_mask(apo_native, erode_kernel=ERODE_KERNEL)
        style_guard = "line"
        guard_geo = apo_geometry_from_mask(bmask, style_guard)
    else:
        bmask = apo_native
        style_guard = style_base
        guard_geo = apo_geometry_from_mask(apo_native, style_guard)

    rows.append(
        {
            "image_id": path.name,
            "res": f"{h}x{w}",
            "pred_cov": pred_cov,
            "style_base": style_base,
            "style_guard": style_guard,
            "guard_applied": bool(guard_applied),
            "mt_ok_base": bool(not np.isnan(base_geo["mt_px"])),
            "mt_ok_guard": bool(not np.isnan(guard_geo["mt_px"])),
            "mt_fail_reason_base": base_geo["mt_fail_reason"],
            "mt_fail_reason_guard": guard_geo["mt_fail_reason"],
            "mt_px_base": float(base_geo["mt_px"]) if not np.isnan(base_geo["mt_px"]) else np.nan,
            "mt_px_guard": float(guard_geo["mt_px"]) if not np.isnan(guard_geo["mt_px"]) else np.nan,
            "n_contours_base": base_geo.get("n_contours"),
            "n_contours_guard": guard_geo.get("n_contours"),
        }
    )

df = pd.DataFrame(rows)
out_csv = "/kaggle/working/apo_geometry_guard_compare.csv"
df.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

print()
print("=== MT OK rates ===")
print("base mt_ok:", float(df.mt_ok_base.mean()))
print("guard mt_ok:", float(df.mt_ok_guard.mean()))

print()
print("=== Fail reason counts (base) ===")
print(df.loc[~df.mt_ok_base, "mt_fail_reason_base"].value_counts().to_dict())

print()
print("=== Fail reason counts (guard) ===")
print(df.loc[~df.mt_ok_guard, "mt_fail_reason_guard"].value_counts().to_dict())

fixed = df[(~df.mt_ok_base) & (df.mt_ok_guard)].copy()
print()
print(f"MT fixed count: {len(fixed)}")

# Gallery: show a few fixed cases
if len(fixed) > 0:
    ids = fixed.image_id.tolist()
    rng = random.Random(RANDOM_SEED)
    picks = rng.sample(ids, min(N_GALLERY, len(ids)))
    for i, image_id in enumerate(picks, start=1):
        p = TEST_DIR / image_id
        img_native = load_gray(p)
        h, w = img_native.shape

        pil = open_rgb_256(img_native)
        _, apo_t, _ = apo_learn.predict(pil)
        apo_native = resize_mask_to(tensor_to_mask(apo_t), h, w)

        pred_cov = float(apo_native.mean())
        style_base = tag_apo_style(pred_cov)
        base_geo = apo_geometry_from_mask(apo_native, style_base)

        guard_applied = pred_cov >= PRED_COV_GUARD_THRESH
        if guard_applied:
            bmask = boundary_from_mask(apo_native, erode_kernel=ERODE_KERNEL)
            guard_geo = apo_geometry_from_mask(bmask, "line")
        else:
            bmask = apo_native
            guard_geo = apo_geometry_from_mask(apo_native, style_base)

        inv_base = invert_mask(apo_native)  # for visual comparison only
        inv_guard = invert_mask(bmask)

        fig, axes = plt.subplots(1, 6, figsize=(28, 4.8))
        axes = axes.reshape(-1)

        # image
        axes[0].imshow(img_native, cmap="gray")
        axes[0].set_title("image", fontsize=9)
        axes[0].axis("off")

        # base
        axes[1].imshow(apo_native, cmap="gray", vmin=0, vmax=1)
        axes[1].set_title(f"base pred\n{style_base}", fontsize=9)
        axes[1].axis("off")
        axes[2].imshow(inv_base, cmap="gray", vmin=0, vmax=1)
        axes[2].set_title("base inv", fontsize=9)
        axes[2].axis("off")
        axes[3].imshow(overlay(img_native, apo_native))
        axes[3].set_title("base overlay", fontsize=9)
        axes[3].axis("off")

        # guard
        axes[4].imshow(bmask, cmap="gray", vmin=0, vmax=1)
        axes[4].set_title(f"guard mask\n{style_base if not guard_applied else 'line'}", fontsize=9)
        axes[4].axis("off")
        axes[5].imshow(overlay(img_native, bmask))
        axes[5].set_title("guard overlay", fontsize=9)
        axes[5].axis("off")

        plt.suptitle(
            f"[fixed {i}/{len(picks)}] {image_id} pred_cov={pred_cov:.3f} "
            f"base mt={base_geo['mt_px'] if not np.isnan(base_geo['mt_px']) else 'NaN'} "
            f"guard mt={guard_geo['mt_px'] if not np.isnan(guard_geo['mt_px']) else 'NaN'}",
            fontsize=11,
            y=1.02,
        )
        plt.tight_layout()
        fig_path = FIG_DIR / f"fixed_guard_{i:02d}_{image_id.replace('.tif','')}.png"
        fig.savefig(fig_path, dpi=120, bbox_inches="tight")
        plt.show()
        plt.close(fig)
else:
    print("No MT-fixed cases found; guard likely didn't rescue any.")
